# Inspection des données du robot (LiDAR + Caméra + Commandes)

Ce notebook permet de visualiser frame par frame les données enregistrées dans `dataset_webots.hdf5`.
Il affiche :
1. Le scan LiDAR (vue de dessus)
2. L'image caméra (reconstruite à partir des secteurs HSV si disponible, ou raw)
3. Les commandes moteurs (Brutes et Normalisées)
4. Les capteurs de proximité

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
import pickle
import os

FILENAME = 'dataset_webots.hdf5'

print(f"Chargement de '{FILENAME}'...")

# 1. Chargement des données
with h5py.File(FILENAME, 'r') as f:
    # thymio_prox : Entrées capteurs [0, 1] (déjà normalisé par le robot)
    thymio_prox = f['thymio_prox'][:]
    
    # thymio_commands : Commandes moteurs [-2, 2] (à normaliser)
    thymio_commands = f['thymio_commands'][:]

print(f"Dataset chargé : {len(thymio_commands)} échantillons")
print(f"Exemple commandes brutes : {thymio_commands[0]}")

# 2. Normalisation des commandes moteurs
# L'objectif est [-1, 1]. Les données actuelles sont [-2, 2].
# On divise par 2.0
y = thymio_commands / 2.0
X = thymio_prox

print(f"Exemple commandes normalisées : {y[0]}")
print(f"Min/Max commandes : {y.min()} / {y.max()}")

# 3. Partitionnement (Train / Test)
# X_train, X_test, y_train, y_test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Taille Train : {X_train.shape[0]}")
print(f"Taille Test  : {X_test.shape[0]}")

## 9c. Entraînement du MLP (Réseau à deux couches cachées)

L'énoncé demande un réseau avec **deux couches cachées**. Nous allons utiliser `MLPRegressor` car nous prédisons des valeurs continues (vitesses moteurs) et non des classes.

Nous allons configurer :
- **Hidden Layers** : `(10, 10)` (exemple de départ).
- **Activation** : `tanh`
- **Solver** : `adam` (efficace pour les grands datasets).
- **Max Iter** : 500 (pour laisser le temps de converger).

In [ ]:
# 4. Configuration et Entraînement
mlp = MLPRegressor(
    hidden_layer_sizes=(10, 10), # 2 couches cachées de 10 neurones
    activation='tanh', 
    solver='adam', 
    max_iter=500,
    random_state=42,
    verbose=True
)

print("Début de l'entraînement...")
mlp.fit(X_train, y_train)
print("Entraînement terminé.")

## 9d. Evaluation

Il faut obtenir un score $R^2 > 0.97$.
Nous calculons le score sur l'ensemble de TEST.

In [ ]:
# 5. Evaluation
score = mlp.score(X_test, y_test)
print(f"Score R^2 sur TEST : {score:.4f}")

# Petit aperçu des prédictions
sample_idx = 100
predicted = mlp.predict([X_test[sample_idx]])
truth = y_test[sample_idx]

print(f"Exemple n°{sample_idx} :")
print(f"  Vrai : {truth}")
print(f"  Prédiction : {predicted[0]}")
print(f"  Erreur : {np.abs(predicted[0] - truth)}")

## 9e. Sauvegarde du Modèle

Si le score est satisfaisant (> 0.97), on sauvegarde le modèle dans un fichier `.model` avec Pickle.
Nous allons sauvegarder le modèle qui prédit des valeurs normalisées ([-1, 1]).
Côté robot, il faudra penser à multiplier la sortie par **2.0** pour retrouver la vraie commande.

In [ ]:
# 6. Sauvegarde
filename = 'ai_controller_model_hyper_prox.model'

# On peut définir un seuil plus strict ou plus souple selon la réalité
if score > 0.90: 
    print(f"Score satisfaisant ({score:.4f} > 0.90). Sauvegarde en cours...")
    with open(filename, 'wb') as model_file:
        pickle.dump(mlp, model_file)
    print(f"Modèle sauvegardé dans : {filename}")
else:
    print(f"Score insuffisant ({score:.4f} <= 0.90). Modèle NON sauvegardé.")
    print("Essayez d'augmenter le nombre de neurones ou d'itérations.")